# Deliverable 1: Data Cleaning & Data Dictionary
**Project:** The Role of Data Analytics in Identifying Revenue Collection Leakages at Forbes Border Post, Zimbabwe
**Notebook:** 01 – Data cleaning, data-quality checks and data dictionary

**Objective:** Load the raw simulated ZIMRA-style dataset, check it for quality problems (missing values, duplicates, inconsistent text and arithmetic errors), add the helper columns needed for analysis and Power BI, and produce a data dictionary.

**Why this matters:** Every number in the dashboard and the management report comes from the file this notebook produces. If the data is wrong here, every later finding is wrong too, so each check below is documented together with what it means.

> **Note on the data:** The workbook's own `Read_Me` sheet says all records are *fictional*. It also says `Potential_Revenue_Gap_USD` is an *analytical flag/estimate*, **not proof of actual revenue loss or fraud**. We keep that caveat in all interpretations.

In [1]:
import pandas as pd
import numpy as np
import os

# Set display options to see all columns
pd.set_option('display.max_columns', None)

# Ensure the processed data and reports directories exist
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

print("Libraries imported and directories verified.")

Libraries imported and directories verified.


## 1. Load the raw data
The workbook has three sheets:
* **`Customs_Transactions`**: one row per customs declaration (the main fact table).
* **`Monthly_Targets`**: the monthly revenue target for Forbes. The proposal asks us to *"compare revenue targets against actual collections"*, so this sheet is loaded too. The earlier version of this notebook ignored it.
* **`Read_Me`**: notes from the data provider.

*Correction made:* the notebook lives in `notebooks/`, but the Excel file sits in the project root, so `'Forbes_Border_Post_Simulated_Data.xlsx'` only worked if a copy happened to be next to the notebook. The code below now searches the likely locations.

In [2]:
# Find the raw file whether it sits in the project root, in data/raw, or next to the notebook
candidates = ['../Forbes_Border_Post_Simulated_Data.xlsx',
              '../data/raw/Forbes_Border_Post_Simulated_Data.xlsx',
              'Forbes_Border_Post_Simulated_Data.xlsx']
file_path = next(p for p in candidates if os.path.exists(p))

# Load the data
df = pd.read_excel(file_path, sheet_name='Customs_Transactions')
targets = pd.read_excel(file_path, sheet_name='Monthly_Targets')

print(f"Dataset loaded from: {file_path}")
print(f"Transactions shape: {df.shape[0]} rows and {df.shape[1]} columns.")
print(f"Monthly targets shape: {targets.shape[0]} rows and {targets.shape[1]} columns.")
print(f"Date range: {df['Declaration_Date'].min().date()} to {df['Declaration_Date'].max().date()}")

Dataset loaded from: ../Forbes_Border_Post_Simulated_Data.xlsx
Transactions shape: 480 rows and 24 columns.
Monthly targets shape: 12 rows and 3 columns.
Date range: 2025-01-01 to 2025-12-30


### Interpretation: what the data covers
* There are **480 declarations and 24 columns**, which matches the `Read_Me` sheet.
* The dates run from **1 January 2025 to 30 December 2025**, so the data covers **12 months, not 8**. The README, the management report and the Power BI brief all say *"January–August 2025 / 8-month period"*, and those must be corrected to **January–December 2025 (12 months)**. A wrong period in a management report undermines its credibility.
* The targets sheet has **12 monthly targets** (2025-01 to 2025-12), one for each month of transactions, so revenue performance can be measured against targets for the whole year.

In [3]:
# View the first 5 rows
display(df.head())

# Inspect the last 3 rows (to confirm the file ends cleanly, with no truncated/partial row)
print("\n--- Last 3 rows ---")
display(df.tail(3))

# Check data types and non-null counts
print("\n--- Dataset Info ---")
df.info()

,Declaration_ID,Declaration_Date,Border_Post,Importer_ID,Country_of_Origin,Commodity_Description,HS_Code,Risk_Category,Declared_Customs_Value_USD,Assessed_Customs_Value_USD,Declared_Customs_Duty_USD,Assessed_Customs_Duty_USD,Declared_Import_VAT_USD,Assessed_Import_VAT_USD,Declared_Excise_USD,Assessed_Excise_USD,Declared_Total_Revenue_USD,Assessed_Total_Revenue_USD,Inspection_Conducted,Offence_Detected,Offence_Type,Potential_Revenue_Gap_USD,Additional_Revenue_Recovered_USD,Payment_Status
0,FORB-0074,2025-01-01,Forbes,Importer_055,Mozambique,Plastic products,3926,Medium,4659.10,4659.10,326.14,326.14,747.79,747.79,0.0,0.0,1073.93,1073.93,No,No,NaN,0.00,0.0,Paid
1,FORB-0362,2025-01-01,Forbes,Importer_048,Mozambique,Solar equipment,8541,Low,10478.80,10478.80,1257.46,1257.46,1760.44,1760.44,0.0,0.0,3017.90,3017.90,No,No,NaN,0.00,0.0,Paid
2,FORB-0341,2025-01-04,Forbes,Importer_021,Zambia,Vehicle spare parts,8708,High,2991.93,2991.93,448.79,448.79,516.11,516.11,0.0,0.0,964.90,964.90,Yes,No,NaN,0.00,0.0,Paid
3,FORB-0039,2025-01-04,Forbes,Importer_049,China,Textiles and clothing,6204,Low,2094.00,2698.08,251.28,323.77,351.79,453.28,0.0,0.0,603.07,777.05,No,No,NaN,173.98,0.0,Paid
4,FORB-0428,2025-01-04,Forbes,Importer_022,South Africa,Agricultural inputs,3105,High,5231.48,6519.98,523.15,652.00,863.19,1075.80,0.0,0.0,1386.34,1727.80,No,No,NaN,341.46,0.0,Paid



--- Last 3 rows ---


,Declaration_ID,Declaration_Date,Border_Post,Importer_ID,Country_of_Origin,Commodity_Description,HS_Code,Risk_Category,Declared_Customs_Value_USD,Assessed_Customs_Value_USD,Declared_Customs_Duty_USD,Assessed_Customs_Duty_USD,Declared_Import_VAT_USD,Assessed_Import_VAT_USD,Declared_Excise_USD,Assessed_Excise_USD,Declared_Total_Revenue_USD,Assessed_Total_Revenue_USD,Inspection_Conducted,Offence_Detected,Offence_Type,Potential_Revenue_Gap_USD,Additional_Revenue_Recovered_USD,Payment_Status
477,FORB-0357,2025-12-28,Forbes,Importer_027,South Africa,Furniture,9403,High,3963.39,3963.39,317.07,317.07,642.07,642.07,0.0,0.0,959.14,959.14,Yes,No,NaN,0.00,0.0,Paid
478,FORB-0179,2025-12-29,Forbes,Importer_054,India,Agricultural inputs,3105,Low,2330.41,3007.00,233.04,300.70,384.52,496.15,0.0,0.0,617.56,796.85,No,No,NaN,179.29,0.0,Paid
479,FORB-0444,2025-12-30,Forbes,Importer_019,Zambia,Furniture,9403,Medium,1732.89,1732.89,138.63,138.63,280.73,280.73,0.0,0.0,419.36,419.36,No,No,NaN,0.00,0.0,Paid



--- Dataset Info ---
<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 24 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Declaration_ID                    480 non-null    str           
 1   Declaration_Date                  480 non-null    datetime64[us]
 2   Border_Post                       480 non-null    str           
 3   Importer_ID                       480 non-null    str           
 4   Country_of_Origin                 480 non-null    str           
 5   Commodity_Description             480 non-null    str           
 6   HS_Code                           480 non-null    int64         
 7   Risk_Category                     480 non-null    str           
 8   Declared_Customs_Value_USD        480 non-null    float64       
 9   Assessed_Customs_Value_USD        480 non-null    float64       
 10  Declared_Customs_Duty_USD         480 n

### Interpretation: first look
* **The last rows are complete.** The earlier notebook assumed *"the last row is truncated"*, but every column in the final rows is filled in, and the next step confirms that no row is dropped. The README's mention of *"truncated rows"* should be reworded as *"checked for truncated rows; none found"*.
* **Only one column has missing values: `Offence_Type`** (25 non-null out of 480). This is **structural, not an error**: an offence type only exists when an offence was found. The 455 blanks mean *"no offence"*, so they are filled with that label, not dropped.
* Data types are mostly correct already: the date is a real date, the money columns are numeric and the text columns are text. `HS_Code` is stored as a number. That is acceptable here, but in Power BI set it to **Text**, because HS codes are labels, not quantities you would add up.

## 2. Cleaning: duplicates, text standardisation, missing values and data types
The proposal lists these cleaning tasks: *remove duplicates, correct data types, standardise dates, handle missing values, and correct inconsistencies in commodity descriptions and tariff codes.* Each one is handled below.

In [4]:
# 1. Drop any incomplete rows (safety check: rows missing their ID or assessed revenue)
initial_row_count = len(df)
df = df.dropna(subset=['Declaration_ID', 'Assessed_Total_Revenue_USD'])
print(f"Incomplete rows dropped: {initial_row_count - len(df)}")

# 2. Check for duplicate Declaration IDs and fully duplicated rows
dup_ids = df.duplicated(subset=['Declaration_ID']).sum()
dup_rows = df.duplicated().sum()
print(f"Duplicate Declaration IDs found: {dup_ids} | Fully duplicated rows: {dup_rows}")
if dup_ids > 0:
    df = df.drop_duplicates(subset=['Declaration_ID'], keep='first')
    print("Duplicates removed.")

# 3. Standardise text: trim stray spaces so 'Tyres ' and 'Tyres' are not counted as two products
text_cols = ['Declaration_ID', 'Border_Post', 'Importer_ID', 'Country_of_Origin',
             'Commodity_Description', 'Risk_Category', 'Inspection_Conducted',
             'Offence_Detected', 'Offence_Type', 'Payment_Status']
for col in text_cols:
    df[col] = df[col].where(df[col].isna(), df[col].astype(str).str.strip())

# 4. Check commodity descriptions against tariff (HS) codes: each commodity should map to exactly one HS code
hs_map = df.groupby('Commodity_Description')['HS_Code'].nunique()
print(f"Commodities with more than one HS code: {(hs_map > 1).sum()}")
print(f"Distinct commodities: {df['Commodity_Description'].nunique()} | Distinct HS codes: {df['HS_Code'].nunique()}")

# 5. Handle missing values BEFORE changing data types
#    A blank Offence_Type means no offence was found, so we label it rather than drop it.
print(f"Blank Offence_Type values: {df['Offence_Type'].isna().sum()} "
      f"(rows with Offence_Detected = 'No': {(df['Offence_Detected'] == 'No').sum()})")
df['Offence_Type'] = df['Offence_Type'].fillna('No Offence')

# 6. Convert data types
df['Declaration_Date'] = pd.to_datetime(df['Declaration_Date'])
categorical_cols = ['Border_Post', 'Country_of_Origin', 'Risk_Category',
                    'Inspection_Conducted', 'Offence_Detected', 'Offence_Type', 'Payment_Status']
for col in categorical_cols:
    df[col] = df[col].astype('category')

# Give Risk_Category a logical order (Low < Medium < High) so tables and charts sort sensibly
df['Risk_Category'] = df['Risk_Category'].cat.reorder_categories(['Low', 'Medium', 'High'], ordered=True)

print(f"\nCleaning complete. Shape: {df.shape[0]} rows and {df.shape[1]} columns.")

Incomplete rows dropped: 0
Duplicate Declaration IDs found: 0 | Fully duplicated rows: 0
Commodities with more than one HS code: 0
Distinct commodities: 10 | Distinct HS codes: 10
Blank Offence_Type values: 455 (rows with Offence_Detected = 'No': 455)

Cleaning complete. Shape: 480 rows and 24 columns.


### Interpretation: cleaning results
* **No incomplete rows and no duplicates.** All 480 declarations are unique, so no transaction is double-counted in the revenue totals.
* **Commodity descriptions and HS codes are consistent.** Each of the 10 commodities maps to exactly one HS code (for example, Tyres → 4011), so the *"inconsistencies in tariff codes"* step in the proposal needed no corrections. That should be stated in the report: we checked, and the data was clean.
* **The 455 blank `Offence_Type` values match the 455 declarations where no offence was detected.** Filling them with *"No Offence"* is therefore correct, not a guess.
* `Risk_Category` is now **ordered Low → Medium → High**, so every table and chart shows the risk levels in their natural order rather than alphabetically.

## 3. Data-quality checks: does the arithmetic add up?
For a revenue-leakage study, the financial columns must be internally consistent. Three rules should hold for every row:
1. **Declared total** = declared duty + declared VAT + declared excise
2. **Assessed total** = assessed duty + assessed VAT + assessed excise
3. **Potential revenue gap** = assessed total − declared total

The earlier notebook did not test these rules. The check is added here.

In [5]:
tol = 0.05  # allow for 5-cent rounding

decl_diff = (df['Declared_Customs_Duty_USD'] + df['Declared_Import_VAT_USD'] + df['Declared_Excise_USD']
             - df['Declared_Total_Revenue_USD'])
ass_diff = (df['Assessed_Customs_Duty_USD'] + df['Assessed_Import_VAT_USD'] + df['Assessed_Excise_USD']
            - df['Assessed_Total_Revenue_USD'])
gap_diff = (df['Assessed_Total_Revenue_USD'] - df['Declared_Total_Revenue_USD']
            - df['Potential_Revenue_Gap_USD'])

df['DQ_Assessed_Components_Mismatch'] = np.where(ass_diff.abs() > tol, 'Yes', 'No')
df['DQ_Assessed_Below_Declared'] = np.where(
    df['Assessed_Total_Revenue_USD'] < df['Declared_Total_Revenue_USD'] - tol, 'Yes', 'No')

print(f"Rule 1 – declared components ≠ declared total: {(decl_diff.abs() > tol).sum()} rows")
print(f"Rule 2 – assessed components ≠ assessed total: {(ass_diff.abs() > tol).sum()} rows")
print(f"Rule 3 – gap ≠ assessed − declared:            {(gap_diff.abs() > tol).sum()} rows")
print(f"Negative revenue gaps recorded:                 {(df['Potential_Revenue_Gap_USD'] < 0).sum()} rows")
print(f"Recovered amount larger than the gap:           "
      f"{(df['Additional_Revenue_Recovered_USD'] > df['Potential_Revenue_Gap_USD'] + tol).sum()} rows")

print("\n--- Rows breaking Rule 2 (assessed components do not add up) ---")
display(df.loc[df['DQ_Assessed_Components_Mismatch'] == 'Yes',
               ['Declaration_ID', 'Offence_Type', 'Declared_Total_Revenue_USD', 'Assessed_Total_Revenue_USD',
                'Potential_Revenue_Gap_USD']].assign(Component_Difference=ass_diff.round(2)))

print("\n--- Rows breaking Rule 3 (assessed BELOW declared, but gap recorded as 0) ---")
display(df.loc[df['DQ_Assessed_Below_Declared'] == 'Yes',
               ['Declaration_ID', 'Declared_Total_Revenue_USD', 'Assessed_Total_Revenue_USD',
                'Potential_Revenue_Gap_USD']].assign(Over_Declared_By=(-gap_diff).round(2)))

Rule 1 – declared components ≠ declared total: 0 rows
Rule 2 – assessed components ≠ assessed total: 7 rows
Rule 3 – gap ≠ assessed − declared:            4 rows
Negative revenue gaps recorded:                 0 rows
Recovered amount larger than the gap:           0 rows

--- Rows breaking Rule 2 (assessed components do not add up) ---


,Declaration_ID,Offence_Type,Declared_Total_Revenue_USD,Assessed_Total_Revenue_USD,Potential_Revenue_Gap_USD,Component_Difference
15,FORB-0407,Misclassification,895.43,1009.19,113.76,-71.28
33,FORB-0131,Misclassification,1077.15,1197.95,120.80,-120.80
42,FORB-0148,Incorrect declaration,1487.48,1550.43,62.95,-62.95
121,FORB-0193,Quantity discrepancy,2064.27,2297.28,233.01,-233.01
198,FORB-0480,Incorrect declaration,1743.89,1947.67,203.78,-203.78
207,FORB-0445,Misclassification,15569.83,16117.78,547.95,-547.95
405,FORB-0142,Misclassification,1476.95,1545.38,68.43,-68.43



--- Rows breaking Rule 3 (assessed BELOW declared, but gap recorded as 0) ---


,Declaration_ID,Declared_Total_Revenue_USD,Assessed_Total_Revenue_USD,Potential_Revenue_Gap_USD,Over_Declared_By
288,FORB-0476,761.40,733.53,0.0,27.87
357,FORB-0413,7406.18,7136.78,0.0,269.40
450,FORB-0456,1164.02,1139.76,0.0,24.26
452,FORB-0196,515.57,499.81,0.0,15.76


### Interpretation: data-quality findings (new)
The data is clean but not perfect. **11 of 480 rows (2.3%) have arithmetic issues**:

* **Rule 1 always holds.** What importers declared adds up correctly.
* **7 rows break Rule 2.** In these, the assessed duty + VAT + excise do not add up to the assessed total. **All 7 are offence cases** (misclassification, incorrect declaration and quantity discrepancy). In 6 of the 7, the shortfall equals the gap exactly. In other words, when customs re-assessed these declarations, the *total* was raised but the duty/VAT/excise columns were left at (almost) the declared amounts. The total and the gap still agree with each other (Rule 3 holds for these rows), so **we keep the assessed total as the authoritative figure.** For any analysis that splits revenue into duty vs VAT vs excise, these 7 rows should be treated with caution.
* **4 rows break Rule 3.** Customs assessed *less* than the importer declared (the importer **over-declared** by about $337 in total), but the gap column shows **0** instead of a negative number. So `Potential_Revenue_Gap_USD` is **one-sided**: it only counts under-declaration and ignores over-declaration. That is a sensible design for a leakage study, but the report should state it as a definition.
* **No row recovered more than its gap**, and no gap is negative, so the recovery figures are plausible.

**Implication:** none of these issues changes the headline totals, so the rows are **flagged, not deleted**. Two flag columns (`DQ_Assessed_Components_Mismatch`, `DQ_Assessed_Below_Declared`) let anyone filter them out in Power BI. These checks belong in the *Methodology* and *Limitations* sections of the report, because they show the data was audited rather than taken on trust.

## 4. Derived columns for analysis and Power BI
A few helper columns make the later analysis and the dashboard much simpler:
* `Month`: year-month label (e.g. `2025-03`), used to join to the monthly targets and to plot trends.
* `Has_Revenue_Gap`: Yes/No, whether customs assessed more than was declared. This is the *"gap incidence"*, which is different from whether an offence was *detected*.
* `Gap_Pct_of_Declared`: the gap as a % of declared revenue, a size-neutral measure of how badly a declaration was understated.

In [6]:
df['Month'] = df['Declaration_Date'].dt.to_period('M').astype(str)
df['Has_Revenue_Gap'] = np.where(df['Potential_Revenue_Gap_USD'] > 0, 'Yes', 'No')
df['Gap_Pct_of_Declared'] = (df['Potential_Revenue_Gap_USD'] / df['Declared_Total_Revenue_USD'] * 100).round(2)

# Clean the targets table so it joins to transactions on Month
targets['Month'] = targets['Month'].astype(str).str.strip().str[:7]
targets['Border_Post'] = targets['Border_Post'].astype(str).str.strip()

print("Gap incidence:")
display(df['Has_Revenue_Gap'].value_counts().to_frame('Declarations'))
print(f"Among declarations with a gap, the median understatement is "
      f"{df.loc[df['Has_Revenue_Gap'] == 'Yes', 'Gap_Pct_of_Declared'].median():.1f}% of declared revenue.")
print(f"\nMonths in transactions that have a target: "
      f"{df['Month'].isin(targets['Month']).all()} ({df['Month'].nunique()} months)")

Gap incidence:


,Declarations
Has_Revenue_Gap,
No,357
Yes,123


Among declarations with a gap, the median understatement is 13.3% of declared revenue.

Months in transactions that have a target: True (12 months)


### Interpretation: gap incidence
* **123 of 480 declarations (about 1 in 4) were under-declared** relative to what customs assessed. However, only **25 declarations had an offence formally detected** (see notebook 02). Most under-declarations were therefore **never classed as offences**, and this distinction drives the rest of the analysis.
* When a declaration is understated, the **median understatement is 13.3% of declared revenue**. That is a meaningful share, not a rounding difference. That points to valuation behaviour rather than data noise.
* Every transaction month has a matching target, so the **targets vs actual** comparison in notebook 02 and Power BI covers all 12 months.

## 5. Data dictionary

In [7]:
descriptions = {
    'Declaration_ID': 'Unique identifier for the customs declaration (primary key)',
    'Declaration_Date': 'Date the declaration was submitted',
    'Border_Post': 'The border post where goods entered (Forbes only)',
    'Importer_ID': 'Unique identifier for the importing entity',
    'Country_of_Origin': 'Country where goods were manufactured',
    'Commodity_Description': 'Text description of the goods (one HS code per commodity)',
    'HS_Code': 'Harmonized System tariff classification code (treat as text label)',
    'Risk_Category': 'Customs risk profile assigned before clearance (Low < Medium < High)',
    'Declared_Customs_Value_USD': 'Customs value declared by the importer',
    'Assessed_Customs_Value_USD': 'Customs value assessed by customs',
    'Declared_Customs_Duty_USD': 'Duty declared by the importer',
    'Assessed_Customs_Duty_USD': 'Duty assessed by customs',
    'Declared_Import_VAT_USD': 'Import VAT declared by the importer',
    'Assessed_Import_VAT_USD': 'Import VAT assessed by customs',
    'Declared_Excise_USD': 'Excise duty declared by the importer (0 for non-excisable goods)',
    'Assessed_Excise_USD': 'Excise duty assessed by customs',
    'Declared_Total_Revenue_USD': 'Total revenue declared (duty + VAT + excise)',
    'Assessed_Total_Revenue_USD': 'Total revenue assessed by customs (authoritative total)',
    'Inspection_Conducted': 'Whether a physical inspection took place (Yes/No)',
    'Offence_Detected': 'Whether an offence was found (Yes/No); only possible when inspected',
    'Offence_Type': "Category of offence; 'No Offence' where none was detected",
    'Potential_Revenue_Gap_USD': 'Assessed minus declared total revenue, floored at 0 (estimate, not proof of loss)',
    'Additional_Revenue_Recovered_USD': 'Revenue actually recovered following a detected offence',
    'Payment_Status': 'Status of the payment (Paid, Part-paid, Pending)',
    'DQ_Assessed_Components_Mismatch': 'Data-quality flag: assessed duty+VAT+excise does not equal assessed total',
    'DQ_Assessed_Below_Declared': 'Data-quality flag: assessed < declared (over-declaration), gap recorded as 0',
    'Month': 'Year-month of the declaration (YYYY-MM); joins to Monthly_Targets',
    'Has_Revenue_Gap': 'Yes if Potential_Revenue_Gap_USD > 0',
    'Gap_Pct_of_Declared': 'Potential revenue gap as % of declared total revenue',
}

data_dict = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.values.astype(str),
    'Non-Null Count': df.notnull().sum().values,
    'Unique Values': [df[col].nunique() for col in df.columns],
    'Description': [descriptions.get(col, 'No description provided') for col in df.columns]
})

print("Data Dictionary generated successfully.")
display(data_dict)

Data Dictionary generated successfully.


,Column Name,Data Type,Non-Null Count,Unique Values,Description
0,Declaration_ID,str,480,480,Unique identifier for the customs declaration ...
1,Declaration_Date,datetime64[us],480,262,Date the declaration was submitted
2,Border_Post,category,480,1,The border post where goods entered (Forbes only)
3,Importer_ID,str,480,60,Unique identifier for the importing entity
4,Country_of_Origin,category,480,6,Country where goods were manufactured
5,Commodity_Description,str,480,10,Text description of the goods (one HS code per...
6,HS_Code,int64,480,10,Harmonized System tariff classification code (...
7,Risk_Category,category,480,3,Customs risk profile assigned before clearance...
8,Declared_Customs_Value_USD,float64,480,480,Customs value declared by the importer
9,Assessed_Customs_Value_USD,float64,480,480,Customs value assessed by customs


### Interpretation: the data dictionary
* Every column is now **fully populated** (480 non-null), and the dictionary records both the original 24 fields and the 5 new ones, so a reader of the dashboard knows exactly what each field means.
* Some *Unique Values* figures are informative on their own: **60 importers** across **6 countries** and **10 commodities**, with only **one border post**. Comparisons between border posts are therefore not possible with this dataset, which is a limitation worth stating.
* `Excise` has few distinct values because most goods are not excisable (the value is 0).

## 6. Save outputs

In [8]:
# 1. Save the cleaned dataset
cleaned_data_path = '../data/processed/cleaned_customs_transactions.csv'
df.to_csv(cleaned_data_path, index=False)
print(f"Cleaned dataset saved to: {cleaned_data_path}")

# 2. Save the cleaned monthly targets (for the targets-vs-actual analysis and Power BI)
targets_path = '../data/processed/monthly_targets.csv'
targets.to_csv(targets_path, index=False)
print(f"Monthly targets saved to: {targets_path}")

# 3. Save the Data Dictionary
dict_path = '../reports/data_dictionary.csv'
data_dict.to_csv(dict_path, index=False)
print(f"Data Dictionary saved to: {dict_path}")

Cleaned dataset saved to: ../data/processed/cleaned_customs_transactions.csv
Monthly targets saved to: ../data/processed/monthly_targets.csv
Data Dictionary saved to: ../reports/data_dictionary.csv


## Summary of Deliverable 1
| Check | Result | Action |
|---|---|---|
| Rows / columns | 480 × 24 raw → 480 × 29 cleaned | 5 helper columns added |
| Period covered | **Jan–Dec 2025 (12 months)** | Fix "Jan–Aug / 8 months" in README, report and dashboard brief |
| Truncated / incomplete rows | None | None needed |
| Duplicate IDs / rows | None | None needed |
| Commodity ↔ HS code consistency | 1-to-1 for all 10 commodities | None needed |
| Missing values | 455 blank `Offence_Type` (structural) | Filled with "No Offence" |
| Assessed components ≠ assessed total | 7 rows (all offence cases) | Flagged; assessed total kept as authoritative |
| Over-declarations with gap set to 0 | 4 rows (~$337) | Flagged; gap defined as one-sided |
| Monthly targets | 12 months, all matched | Saved as `monthly_targets.csv` |

**Bottom line:** the dataset is reliable enough for the leakage analysis. The headline totals are unaffected by the 11 flagged rows. The cleaned files are ready for the exploratory analysis (notebook 02) and Power BI.